In [ ]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
DRIVE_SAVE_PATH = '/content/drive/MyDrive/STUFF/intern_work_video'


In [ ]:
model_path = "yolo26x.pt"


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import os
from google.colab import drive

from scripts.video_helpers import VideoNotebookUtils

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

video_path_input = '/content/intern_video_small.mp4'

if 'DRIVE_SAVE_PATH' not in globals():
    DRIVE_SAVE_PATH = '/content/drive/MyDrive/STUFF/intern_work_video'
if 'model_path' not in globals():
    model_path = 'yolo26x.pt'

values_data = {"sample_image.png":{"fileref":"","size":3434687,"filename":"sample_image.png","base64_img_data":"","file_attributes":{},"regions":{"0":{"shape_attributes":{"name":"polygon","all_points_x":[79.48051948051949,673.2467532467533,1217.142857142857,1901.2987012987014,930.3896103896104,79.48051948051949],"all_points_y":[870.3896103896104,45.97402597402598,45.97402597402598,871.948051948052,968.5714285714287,870.3896103896104]},"region_attributes":{"label":"boundary"}}}}}
polygon_points_for_draw_and_test = VideoNotebookUtils.polygon_from_via(values_data, 'sample_image.png', '0')

model = YOLO(model_path)
pose_model = YOLO('yolov8s-pose.pt')

if not os.path.exists(DRIVE_SAVE_PATH):
    os.makedirs(DRIVE_SAVE_PATH)

output_video_path = os.path.join(DRIVE_SAVE_PATH, 'output_video_with_filtered_detections.mp4')

cap_input = cv2.VideoCapture(video_path_input)
if not cap_input.isOpened():
    raise FileNotFoundError(f"Input video file not found: {video_path_input}")

fps = int(cap_input.get(cv2.CAP_PROP_FPS))
frame_width = int(cap_input.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap_input.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_input.release()

writer = None
for fourcc_str in ['avc1', 'H264', 'mp4v']:
    fourcc = cv2.VideoWriter_fourcc(*fourcc_str)
    writer_try = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    if writer_try.isOpened():
        writer = writer_try
        print(f"VideoWriter codec: {fourcc_str}")
        break

if writer is None:
    raise IOError(f"Could not open video writer for {output_video_path}")

print(f"Processing → {output_video_path}")

for i, res in enumerate(model.predict(video_path_input, stream=True, verbose=False)):
    orig = res.orig_img
    frame = orig.copy()

    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon_points_for_draw_and_test], (0, 255, 0))
    frame = cv2.addWeighted(frame, 0.85, overlay, 0.15, 0)
    cv2.polylines(
        frame,
        [polygon_points_for_draw_and_test],
        isClosed=True,
        color=(0, 255, 0),
        thickness=3,
    )

    pose_res = pose_model.predict(orig, verbose=False)[0]
    if pose_res.keypoints is not None and pose_res.boxes is not None:
        kpts_xy_all = pose_res.keypoints.xy.cpu().numpy()
        kpts_cf_all = pose_res.keypoints.conf.cpu().numpy()
        person_boxes = pose_res.boxes.xyxy.cpu().numpy()

        for j in range(len(person_boxes)):
            pbox = person_boxes[j]
            if not VideoNotebookUtils.all_corners_inside(pbox, polygon_points_for_draw_and_test):
                continue

            x1, y1, x2, y2 = pbox.astype(int)
            kpts_xy = kpts_xy_all[j]
            kpts_cf = kpts_cf_all[j]

            VideoNotebookUtils.draw_pose(frame, kpts_xy, kpts_cf, color=(0, 255, 255), conf_th=0.25)
            shot = VideoNotebookUtils.classify_shot_from_pose(kpts_xy, kpts_cf, conf_th=0.25)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(
                frame,
                shot,
                (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2,
                cv2.LINE_AA,
            )

    if res.boxes is not None:
        for box in res.boxes:
            if not VideoNotebookUtils.all_corners_inside(
                box.xyxy[0].cpu().numpy(),
                polygon_points_for_draw_and_test,
            ):
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id] if model.names else "object"
            color = VideoNotebookUtils.get_class_color(cls_id)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            tag = f"{label} {conf:.2f}"
            (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
            cv2.putText(
                frame,
                tag,
                (x1 + 2, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
            )

            print(f"Frame {i} | {label} {conf:.2f} | box=[{x1},{y1},{x2},{y2}]")

    writer.write(frame)
    if i % 100 == 0:
        print(f"Processed {i} frames...")

writer.release()
print(f"Done. Saved to: {output_video_path}")


Streaming output truncated to the last 5000 lines.
Frame 653 | sports ball 0.37 | box=[1023,272,1033,282]
Frame 654 | person 0.89 | box=[1107,49,1140,125]
Frame 654 | person 0.87 | box=[827,55,864,138]
Frame 654 | person 0.85 | box=[1234,279,1301,447]
Frame 654 | person 0.58 | box=[451,741,576,904]
Frame 654 | sports ball 0.37 | box=[1023,272,1033,282]
Frame 654 | tennis racket 0.33 | box=[516,807,634,868]
Frame 654 | sports ball 0.25 | box=[1221,269,1232,279]
Frame 655 | person 0.89 | box=[1107,49,1142,125]
Frame 655 | person 0.87 | box=[825,56,862,138]
Frame 655 | person 0.87 | box=[1236,280,1303,446]
Frame 655 | person 0.55 | box=[451,741,569,904]
Frame 655 | tennis racket 0.40 | box=[515,806,634,869]
Frame 655 | sports ball 0.37 | box=[1023,272,1033,282]
Frame 655 | sports ball 0.25 | box=[1221,269,1232,278]
Frame 656 | person 0.89 | box=[824,56,861,139]
Frame 656 | person 0.88 | box=[1108,50,1142,125]
Frame 656 | person 0.87 | box=[1235,280,1304,446]
Frame 656 | person 0.51 | box=

In [ ]:
import os

import cv2
import numpy as np
from ultralytics import YOLO

from scripts.video_helpers import VideoNotebookUtils

model_detect = YOLO("yolo26x.pt")
model_pose = YOLO("yolov8x-pose.pt")

BALL_CLASS_IDS = {
    cid
    for cid, name in model_detect.names.items()
    if name.lower() in {"ball", "sports ball", "tennis ball", "baseball", "soccer ball"}
}
PLAYER_MIN_CONF = 0.4
BALL_MIN_CONF = 0.3
POSE_MIN_CONF = 0.4

print(f"🎯 Ball class IDs: {BALL_CLASS_IDS}")

values_data = {"sample_image.png":{"fileref":"","size":3434687,"filename":"sample_image.png","base64_img_data":"","file_attributes":{},"regions":{"0":{"shape_attributes":{"name":"polygon","all_points_x":[79.48051948051949,673.2467532467533,1217.142857142857,1901.2987012987014,930.3896103896104,79.48051948051949],"all_points_y":[870.3896104,45.97402597,45.97402597,871.9480519,968.57142857,870.3896104]},"region_attributes":{"label":"boundary"}}}}}
polygon_pts = VideoNotebookUtils.polygon_from_via(values_data, 'sample_image.png', '0')

BALL_COLOR = (0, 255, 255)

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
cap = cv2.VideoCapture(video_path_input)
if not cap.isOpened():
    raise FileNotFoundError(f"Video not found: {video_path_input}")

fps = int(cap.get(cv2.CAP_PROP_FPS))
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

print(f"🎬 Processing: {w}x{h} @ {fps}fps → {output_video_path}")

frame_idx = 0
for detect_res in model_detect.predict(video_path_input, stream=True, verbose=False, conf=0.3):
    frame = detect_res.orig_img.copy()

    pose_res = model_pose.predict(source=frame, conf=POSE_MIN_CONF, verbose=False)[0]

    overlay = frame.copy()
    cv2.fillPoly(overlay, [polygon_pts], (0, 255, 0))
    frame = cv2.addWeighted(frame, 0.85, overlay, 0.15, 0)
    cv2.polylines(frame, [polygon_pts], True, (0, 255, 0), 3)

    if pose_res.keypoints is not None and detect_res.boxes is not None:
        kps_list = pose_res.keypoints.data
        for i, box in enumerate(detect_res.boxes):
            cls_id = int(box.cls[0])
            if cls_id in BALL_CLASS_IDS:
                continue
            if i < len(kps_list):
                kps = kps_list[i].cpu().numpy()
                bbox_xyxy = box.xyxy[0].cpu().numpy()
                VideoNotebookUtils.draw_skeleton(
                    frame,
                    kps,
                    bbox_xyxy,
                    polygon_pts,
                    conf_thresh=POSE_MIN_CONF,
                )

    if detect_res.boxes is not None:
        for box in detect_res.boxes:
            xyxy = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model_detect.names[cls_id]

            if not VideoNotebookUtils.all_corners_inside(xyxy, polygon_pts):
                continue

            x1, y1, x2, y2 = map(int, xyxy)

            if cls_id in BALL_CLASS_IDS and conf >= BALL_MIN_CONF:
                VideoNotebookUtils.draw_box(frame, x1, y1, x2, y2, label, conf, BALL_COLOR)
            elif cls_id not in BALL_CLASS_IDS and conf >= PLAYER_MIN_CONF:
                color = VideoNotebookUtils.get_color(cls_id)
                VideoNotebookUtils.draw_box(frame, x1, y1, x2, y2, label, conf, color)

    out.write(frame)
    if frame_idx % 100 == 0:
        print(f"✅ Frame {frame_idx} processed")
    frame_idx += 1

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"🎉 Done! Output saved to: {output_video_path}")


🎯 Ball class IDs: {32}
🎬 Processing: 1920x1080 @ 25fps → /content/drive/MyDrive/STUFF/intern_work_video/output_video_with_filtered_detections.mp4
✅ Frame 0 processed
✅ Frame 100 processed
✅ Frame 200 processed
✅ Frame 300 processed
✅ Frame 400 processed
✅ Frame 500 processed
✅ Frame 600 processed
✅ Frame 700 processed
✅ Frame 800 processed
✅ Frame 900 processed
✅ Frame 1000 processed
✅ Frame 1100 processed
✅ Frame 1200 processed
✅ Frame 1300 processed
✅ Frame 1400 processed
🎉 Done! Output saved to: /content/drive/MyDrive/STUFF/intern_work_video/output_video_with_filtered_detections.mp4
